#FIFA World Cup 2026 – Regression 2.2

01. Data Preparation
The objective of Regression 2.2 is to build and evaluate a linear regression model that predicts the number of goals scored by a team in a single FIFA World Cup 2026 match.


In [1]:
# Import libraries
import pandas as pd
import numpy as np

# Show all columns
pd.set_option("display.max_columns", None)

In [2]:
# Load the dataset
file_path = "L2_Goal_208.xlsx"

df = pd.read_excel(file_path)

print("Dataset loaded successfully.")

Dataset loaded successfully.


In [3]:
# Check dataset size
print("Dataset shape:", df.shape)

# Check column names
print("\nColumns:")
print(df.columns.tolist())

Dataset shape: (208, 9)

Columns:
['MatchID', 'Date', 'Match', 'Team', 'Opponent', 'Goals_For', 'Goals_Against', 'Result', 'Ball_Possession_pct']


In [4]:
# View first 10 rows
df.head(10)

,MatchID,Date,Match,Team,Opponent,Goals_For,Goals_Against,Result,Ball_Possession_pct
0,1,2026-06-11,Mexico vs South Africa,Mexico,South Africa,2,0,Win,61
1,1,2026-06-11,Mexico vs South Africa,South Africa,Mexico,0,2,Loss,40
2,2,2026-06-11,Korea Republic vs Czechia,Korea Republic,Czechia,2,1,Win,62
3,2,2026-06-11,Korea Republic vs Czechia,Czechia,Korea Republic,1,2,Loss,38
4,3,2026-06-12,Canada vs Bosnia–Herz,Canada,Bosnia–Herz,1,1,Draw,61
5,3,2026-06-12,Canada vs Bosnia–Herz,Bosnia–Herz,Canada,1,1,Draw,39
6,4,2026-06-12,USA vs Paraguay,USA,Paraguay,4,1,Win,65
7,4,2026-06-12,USA vs Paraguay,Paraguay,USA,1,4,Loss,35
8,5,2026-06-13,Qatar vs Switzerland,Qatar,Switzerland,1,1,Draw,32
9,5,2026-06-13,Qatar vs Switzerland,Switzerland,Qatar,1,1,Draw,68


In [5]:
# Check number of rows and matches
print("Number of observations:", len(df))
print("Number of unique matches:", df["MatchID"].nunique())

# Check each match has 2 rows
print("\nRows per MatchID:")
print(df.groupby("MatchID").size().value_counts())

Number of observations: 208
Number of unique matches: 104

Rows per MatchID:
2    104
Name: count, dtype: int64


In [6]:
# Check missing values
print("Missing values:")
print(df.isnull().sum())

# Check duplicate rows
print("\nDuplicate rows:")
print(df.duplicated().sum())

Missing values:
MatchID                0
Date                   0
Match                  0
Team                   0
Opponent               0
Goals_For              0
Goals_Against          0
Result                 0
Ball_Possession_pct    0
dtype: int64

Duplicate rows:
0


In [7]:
# Check data types
df.dtypes

MatchID                         int64
Date                   datetime64[us]
Match                             str
Team                              str
Opponent                          str
Goals_For                       int64
Goals_Against                   int64
Result                            str
Ball_Possession_pct             int64
dtype: object

In [8]:
# Check number of team names
print("Raw number of unique team names:", df["Team"].nunique())

# Show all team names
print("\nTeam names:")
print(sorted(df["Team"].unique()))

Raw number of unique team names: 50

Team names:
['Algeria', 'Argentina', 'Australia', 'Austria', 'Belgium', 'Bosnia–Herz', 'Brazil', 'Cabo Verde', 'Canada', 'Colombia', 'Congo DR', 'Croatia', 'Curaçao', 'Czechia', "Côte d'Ivoire", 'Ecuador', 'Egypt', 'England', 'England\xa0', 'France', 'Germany', 'Ghana', 'Haiti', 'IR Iran', 'Iraq', 'Japan', 'Jordan', 'Korea Republic', 'Mexico', 'Morocco', 'Netherlands', 'New Zealand', 'Norway', 'Panama', 'Paraguay', 'Portugal', 'Qatar', 'Saudi Arabia', 'Scotland', 'Scotland\xa0', 'Senegal', 'South Africa', 'Spain', 'Sweden', 'Switzerland', 'Tunisia', 'Türkiye', 'USA', 'Uruguay', 'Uzbekistan']


In [9]:
# Check for hidden spaces in team names
suspicious_teams = [
    repr(team)
    for team in df["Team"].unique()
    if str(team) != str(team).strip()
    or "\xa0" in str(team)
]

print("Team names with hidden spaces:")
print(suspicious_teams)

Team names with hidden spaces:
["'England\\xa0'", "'Scotland\\xa0'"]


# Step 2 – Data Cleaning and Validation

The initial validation identified hidden whitespace characters in some team names.

This step cleans the text fields and checks that the dataset remains consistent after cleaning.

In [10]:
# Create a copy for cleaning
df_clean = df.copy()

In [11]:
# Clean team and opponent names
for col in ["Team", "Opponent"]:
    df_clean[col] = (
        df_clean[col]
        .str.replace("\xa0", " ", regex=False)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

In [12]:
# Clean match names
df_clean["Match"] = (
    df_clean["Match"]
    .str.replace("\xa0", " ", regex=False)
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

In [13]:
# Check team names after cleaning
print("Unique teams after cleaning:", df_clean["Team"].nunique())

suspicious_teams = [
    repr(team)
    for team in df_clean["Team"].unique()
    if str(team) != str(team).strip()
    or "\xa0" in str(team)
]

print("Team names with hidden spaces:", suspicious_teams)

Unique teams after cleaning: 48
Team names with hidden spaces: []


In [14]:
# Check team and opponent names match
team_names = set(df_clean["Team"].unique())
opponent_names = set(df_clean["Opponent"].unique())

print("Teams:", len(team_names))
print("Opponents:", len(opponent_names))
print("Difference:", team_names.symmetric_difference(opponent_names))

Teams: 48
Opponents: 48
Difference: set()


In [15]:
# Check numeric values
print("Negative Goals_For:", (df_clean["Goals_For"] < 0).sum())
print("Negative Goals_Against:", (df_clean["Goals_Against"] < 0).sum())

print(
    "Possession values outside 0–100:",
    (~df_clean["Ball_Possession_pct"].between(0, 100)).sum()
)

Negative Goals_For: 0
Negative Goals_Against: 0
Possession values outside 0–100: 0


In [16]:
# Check result categories
print(df_clean["Result"].value_counts())

print("\nResult types:")
print(df_clean["Result"].unique())

Result
Win     80
Loss    80
Draw    48
Name: count, dtype: int64

Result types:
<StringArray>
['Win', 'Loss', 'Draw']
Length: 3, dtype: str


In [17]:
# Check scores match between both rows
score_errors = 0

for match_id, group in df_clean.groupby("MatchID"):
    row1 = group.iloc[0]
    row2 = group.iloc[1]

    if not (
        row1["Goals_For"] == row2["Goals_Against"]
        and row1["Goals_Against"] == row2["Goals_For"]
    ):
        score_errors += 1

print("Matches with score inconsistencies:", score_errors)


Matches with score inconsistencies: 0


In [18]:
# Check result matches the score
def expected_result(row):
    if row["Goals_For"] > row["Goals_Against"]:
        return "Win"
    elif row["Goals_For"] < row["Goals_Against"]:
        return "Loss"
    else:
        return "Draw"

result_errors = (
    df_clean.apply(expected_result, axis=1)
    != df_clean["Result"]
).sum()

print("Rows with incorrect Result:", result_errors)

Rows with incorrect Result: 0


In [19]:
# Final check after cleaning
print("Dataset shape:", df_clean.shape)
print("Missing values:", df_clean.isnull().sum().sum())
print("Duplicate rows:", df_clean.duplicated().sum())
print("Unique teams:", df_clean["Team"].nunique())
print("Unique matches:", df_clean["MatchID"].nunique())

Dataset shape: (208, 9)
Missing values: 0
Duplicate rows: 0
Unique teams: 48
Unique matches: 104


# Data Cleaning Summary

Hidden whitespace characters were removed from the team, opponent and match-name fields. After cleaning, the dataset contained 48 unique teams, matching the expected number of participants in the FIFA World Cup 2026.

The team and opponent names were consistent, all goal and possession values were within valid ranges, and the match scores and result labels were internally consistent.

The cleaned dataset retained all 208 observations and 104 matches, with no missing values or duplicate rows.

# Step 3 – Define Variables

The target variable is `Goals_For`.

Eight pre-match variables are used to predict how many goals a team scores.

In [20]:
# Define target
target = "Goals_For"

# Define 8 predictors
features = [
    "Team_PreTournament_FIFA_Rank",
    "Opponent_PreTournament_FIFA_Rank",
    "Team_PreMatch_Goals_PG",
    "Opponent_PreMatch_GoalsAgainst_PG",
    "Team_PreMatch_GoalsAgainst_PG",
    "Opponent_PreMatch_Points_PG",
    "Team_PreMatch_Possession_Avg",
    "Team_PreMatch_Points_PG"
]

print("Target:", target)
print("Number of predictors:", len(features))

for feature in features:
    print("-", feature)

Target: Goals_For
Number of predictors: 8
- Team_PreTournament_FIFA_Rank
- Opponent_PreTournament_FIFA_Rank
- Team_PreMatch_Goals_PG
- Opponent_PreMatch_GoalsAgainst_PG
- Team_PreMatch_GoalsAgainst_PG
- Opponent_PreMatch_Points_PG
- Team_PreMatch_Possession_Avg
- Team_PreMatch_Points_PG


# Data Leakage

Only information available before each match is used. Statistics from the current match are not used as predictors.

In [21]:
# Confirm exactly 8 predictors
assert len(features) == 8

print("Exactly 8 explanatory variables selected.")

Exactly 8 explanatory variables selected.


# Step 4 – Feature Engineering

Pre-match statistics are calculated using only matches played before the current match.

In [22]:
# Start from cleaned data
df_model = df_clean.copy()

# Put matches in tournament order
df_model = df_model.sort_values(
    "MatchID",
    kind="stable"
).reset_index(drop=True)

print("Rows:", len(df_model))

Rows: 208


In [23]:
# Convert results into points
points_map = {
    "Win": 3,
    "Draw": 1,
    "Loss": 0
}

df_model["Points"] = df_model["Result"].map(points_map)

# Count previous matches
df_model["Previous_Matches"] = (
    df_model.groupby("Team").cumcount()
)

print("Missing points:", df_model["Points"].isnull().sum())

Missing points: 0


In [24]:
# X3: previous average goals scored
df_model["Team_PreMatch_Goals_PG"] = (
    df_model.groupby("Team")["Goals_For"]
    .transform(lambda x: x.shift(1).expanding().mean())
)

# X5: previous average goals conceded
df_model["Team_PreMatch_GoalsAgainst_PG"] = (
    df_model.groupby("Team")["Goals_Against"]
    .transform(lambda x: x.shift(1).expanding().mean())
)

# X7: previous average possession
df_model["Team_PreMatch_Possession_Avg"] = (
    df_model.groupby("Team")["Ball_Possession_pct"]
    .transform(lambda x: x.shift(1).expanding().mean())
)

# X8: previous average points
df_model["Team_PreMatch_Points_PG"] = (
    df_model.groupby("Team")["Points"]
    .transform(lambda x: x.shift(1).expanding().mean())
)

In [25]:
# Create opponent pre-match history
opponent_history = df_model[
    [
        "MatchID",
        "Team",
        "Team_PreMatch_GoalsAgainst_PG",
        "Team_PreMatch_Points_PG"
    ]
].rename(
    columns={
        "Team": "Opponent",
        "Team_PreMatch_GoalsAgainst_PG":
            "Opponent_PreMatch_GoalsAgainst_PG",
        "Team_PreMatch_Points_PG":
            "Opponent_PreMatch_Points_PG"
    }
)

# Add opponent history
df_model = df_model.merge(
    opponent_history,
    on=["MatchID", "Opponent"],
    how="left",
    validate="one_to_one"
)

# Keep tournament order
df_model = df_model.sort_values(
    "MatchID",
    kind="stable"
).reset_index(drop=True)

print("Rows after opponent merge:", len(df_model))

Rows after opponent merge: 208


In [27]:
# Pre-tournament FIFA rankings
fifa_rank = {
    "Argentina": 1,
    "Spain": 2,
    "France": 3,
    "England": 4,
    "Portugal": 5,
    "Brazil": 6,
    "Morocco": 7,
    "Netherlands": 8,
    "Belgium": 9,
    "Germany": 10,
    "Croatia": 11,
    "Colombia": 13,
    "Mexico": 14,
    "Senegal": 15,
    "Uruguay": 16,
    "USA": 17,
    "Japan": 18,
    "Switzerland": 19,
    "IR Iran": 20,
    "Türkiye": 22,
    "Ecuador": 23,
    "Austria": 24,
    "Korea Republic": 25,
    "Australia": 27,
    "Algeria": 28,
    "Egypt": 29,
    "Canada": 30,
    "Norway": 31,
    "Côte d'Ivoire": 33,
    "Panama": 34,
    "Sweden": 38,
    "Czechia": 40,
    "Paraguay": 41,
    "Scotland": 42,
    "Tunisia": 45,
    "Congo DR": 46,
    "Uzbekistan": 50,
    "Qatar": 56,
    "Iraq": 57,
    "South Africa": 60,
    "Saudi Arabia": 61,
    "Jordan": 63,
    "Bosnia–Herz": 64,
    "Cabo Verde": 67,
    "Ghana": 73,
    "Curaçao": 82,
    "Haiti": 83,
    "New Zealand": 85
}

print("FIFA ranking teams:", len(fifa_rank))

FIFA ranking teams: 48


In [38]:
# team FIFA ranking
df_model["Team_PreTournament_FIFA_Rank"] = (
    df_model["Team"].map(fifa_rank)
)

# X2: opponent FIFA ranking
df_model["Opponent_PreTournament_FIFA_Rank"] = (
    df_model["Opponent"].map(fifa_rank)
)

print(
    "Missing team rankings:",
    df_model["Team_PreTournament_FIFA_Rank"].isnull().sum()
)

print(
    "Missing opponent rankings:",
    df_model["Opponent_PreTournament_FIFA_Rank"].isnull().sum()
)

Missing team rankings: 0
Missing opponent rankings: 0


In [39]:
# Check historical features
historical_features = [
    "Team_PreMatch_Goals_PG",
    "Opponent_PreMatch_GoalsAgainst_PG",
    "Team_PreMatch_GoalsAgainst_PG",
    "Opponent_PreMatch_Points_PG",
    "Team_PreMatch_Possession_Avg",
    "Team_PreMatch_Points_PG"
]

print(df_model[historical_features].isnull().sum())

Team_PreMatch_Goals_PG               48
Opponent_PreMatch_GoalsAgainst_PG    48
Team_PreMatch_GoalsAgainst_PG        48
Opponent_PreMatch_Points_PG          48
Team_PreMatch_Possession_Avg         48
Team_PreMatch_Points_PG              48
dtype: int64


In [40]:
# Check all predictors
missing_features = [
    feature for feature in features
    if feature not in df_model.columns
]

print("Number of predictors:", len(features))
print("Missing predictor columns:", missing_features)

print("\nRows:", len(df_model))
print("Unique matches:", df_model["MatchID"].nunique())
print("Unique teams:", df_model["Team"].nunique())

assert len(features) == 8
assert len(missing_features) == 0
assert len(df_model) == 208
assert df_model["MatchID"].nunique() == 104
assert df_model["Team"].nunique() == 48

print("\n feature engineering completed successfully.")

Number of predictors: 8
Missing predictor columns: []

Rows: 208
Unique matches: 104
Unique teams: 48

 feature engineering completed successfully.


# Step 5 – Final Dataset

The final modelling dataset contains exactly eight explanatory variables and the target variable `Goals_For`.

In [41]:
# Select IDs, predictors and target
final_columns = [
    "MatchID",
    "Team",
    "Opponent",
    "Team_PreTournament_FIFA_Rank",
    "Opponent_PreTournament_FIFA_Rank",
    "Team_PreMatch_Goals_PG",
    "Opponent_PreMatch_GoalsAgainst_PG",
    "Team_PreMatch_GoalsAgainst_PG",
    "Opponent_PreMatch_Points_PG",
    "Team_PreMatch_Possession_Avg",
    "Team_PreMatch_Points_PG",
    "Goals_For"
]

final_model = df_model[final_columns].copy()

final_model.head()

,MatchID,Team,Opponent,Team_PreTournament_FIFA_Rank,Opponent_PreTournament_FIFA_Rank,Team_PreMatch_Goals_PG,Opponent_PreMatch_GoalsAgainst_PG,Team_PreMatch_GoalsAgainst_PG,Opponent_PreMatch_Points_PG,Team_PreMatch_Possession_Avg,Team_PreMatch_Points_PG,Goals_For
0,1,Mexico,South Africa,14,60,NaN,NaN,NaN,NaN,NaN,NaN,2
1,1,South Africa,Mexico,60,14,NaN,NaN,NaN,NaN,NaN,NaN,0
2,2,Korea Republic,Czechia,25,40,NaN,NaN,NaN,NaN,NaN,NaN,2
3,2,Czechia,Korea Republic,40,25,NaN,NaN,NaN,NaN,NaN,NaN,1
4,3,Canada,Bosnia–Herz,30,64,NaN,NaN,NaN,NaN,NaN,NaN,1


In [42]:
# Check final dataset structure
print("Dataset shape:", final_model.shape)
print("Rows:", len(final_model))
print("Unique matches:", final_model["MatchID"].nunique())
print("Unique teams:", final_model["Team"].nunique())

print(
    "Duplicate team-match rows:",
    final_model.duplicated(["MatchID", "Team"]).sum()
)

Dataset shape: (208, 12)
Rows: 208
Unique matches: 104
Unique teams: 48
Duplicate team-match rows: 0


In [43]:
# Check missing predictor values
print(final_model[features].isnull().sum())

print(
    "\nMissing target values:",
    final_model[target].isnull().sum()
)

Team_PreTournament_FIFA_Rank          0
Opponent_PreTournament_FIFA_Rank      0
Team_PreMatch_Goals_PG               48
Opponent_PreMatch_GoalsAgainst_PG    48
Team_PreMatch_GoalsAgainst_PG        48
Opponent_PreMatch_Points_PG          48
Team_PreMatch_Possession_Avg         48
Team_PreMatch_Points_PG              48
dtype: int64

Missing target values: 0


In [44]:
# Check predictor and target data types
model_columns = features + [target]

print(final_model[model_columns].dtypes)

Team_PreTournament_FIFA_Rank           int64
Opponent_PreTournament_FIFA_Rank       int64
Team_PreMatch_Goals_PG               float64
Opponent_PreMatch_GoalsAgainst_PG    float64
Team_PreMatch_GoalsAgainst_PG        float64
Opponent_PreMatch_Points_PG          float64
Team_PreMatch_Possession_Avg         float64
Team_PreMatch_Points_PG              float64
Goals_For                              int64
dtype: object


In [45]:
# Check current-match information is not included
leakage_columns = [
    "Goals_Against",
    "Result",
    "Ball_Possession_pct",
    "Points",
    "Previous_Matches"
]

included_leakage = [
    col for col in leakage_columns
    if col in final_model.columns
]

print("Leakage columns included:", included_leakage)

Leakage columns included: []


In [46]:
# Final validation
historical_features = [
    "Team_PreMatch_Goals_PG",
    "Opponent_PreMatch_GoalsAgainst_PG",
    "Team_PreMatch_GoalsAgainst_PG",
    "Opponent_PreMatch_Points_PG",
    "Team_PreMatch_Possession_Avg",
    "Team_PreMatch_Points_PG"
]

assert final_model.shape == (208, 12)
assert final_model["MatchID"].nunique() == 104
assert final_model["Team"].nunique() == 48
assert final_model.duplicated(["MatchID", "Team"]).sum() == 0
assert len(features) == 8
assert all(feature in final_model.columns for feature in features)
assert final_model[target].isnull().sum() == 0
assert all(final_model[col].isnull().sum() == 48 for col in historical_features)
assert final_model["Team_PreTournament_FIFA_Rank"].isnull().sum() == 0
assert final_model["Opponent_PreTournament_FIFA_Rank"].isnull().sum() == 0
assert len(included_leakage) == 0

print("Final modelling dataset validated successfully.")

Final modelling dataset validated successfully.


In [47]:
# Save prepared dataset
final_model.to_excel(
    "Regression_2_2_Final_Model_Data.xlsx",
    index=False
)

print("Final dataset saved successfully.")

Final dataset saved successfully.


# Final Data Preparation Summary

The final dataset contains 208 team-match observations, 104 matches and exactly eight pre-match explanatory variables.

The missing historical values occur in each team's first tournament match because no previous World Cup 2026 match history was available.

The dataset is now ready for exploratory data analysis and regression modelling.